# Metricas del fine-tuning

Requiere haber corrido `2_finetune.py` primero.

In [ ]:
import json, glob, os
import matplotlib.pyplot as plt

ADAPTER = sorted(glob.glob('adapters/*/metricas.json'))[-1]
hist = json.load(open(ADAPTER))
print('Adaptador:', os.path.dirname(ADAPTER))

## 1. Curvas de loss

Train bajando + eval bajando = aprende. Eval subiendo mientras train baja = overfitting.

In [ ]:
tr = [(h['step'], h['loss']) for h in hist if 'loss' in h]
ev = [(h['step'], h['eval_loss']) for h in hist if 'eval_loss' in h]

plt.figure(figsize=(9, 4))
plt.plot(*zip(*tr), label='train', marker='o', ms=3)
if ev:
    plt.plot(*zip(*ev), label='eval', marker='s', ms=4)
plt.xlabel('step'); plt.ylabel('loss'); plt.legend(); plt.grid(alpha=.3)
plt.title('Loss durante el entrenamiento')
plt.show()

print(f"train: {tr[0][1]:.3f} -> {tr[-1][1]:.3f}  ({100*(1-tr[-1][1]/tr[0][1]):.0f}% de reduccion)")
if ev:
    print(f"eval:  {ev[0][1]:.3f} -> {ev[-1][1]:.3f}")
    print('\nOVERFITTING' if ev[-1][1] > min(e[1] for e in ev) * 1.1 else '\nSin overfitting evidente')

## 2. Resumen del entrenamiento

In [ ]:
fin = hist[-1]
for k in ['train_runtime', 'train_samples_per_second', 'total_flos', 'epoch']:
    if k in fin:
        print(f'{k:28} {fin[k]}')

tam = sum(os.path.getsize(os.path.join(os.path.dirname(ADAPTER), f))
          for f in os.listdir(os.path.dirname(ADAPTER)) if f.endswith('.safetensors'))
print(f'{"tamano adaptador (MB)":28} {tam/1e6:.1f}')

## 3. Probar el modelo

La prueba que importa: ¿habla como Yoda en español?

In [ ]:
import torch
from peft import AutoPeftModelForCausalLM
from transformers import AutoTokenizer

ruta = os.path.dirname(ADAPTER)
model = AutoPeftModelForCausalLM.from_pretrained(ruta, device_map='cuda:0')
tok = AutoTokenizer.from_pretrained(ruta)
model.eval()

def generar(texto):
    msgs = [{'role': 'user', 'content': texto}]
    prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inp = tok(prompt, return_tensors='pt', add_special_tokens=False).to(model.device)
    with torch.no_grad():
        out = model.generate(**inp, max_new_tokens=64, eos_token_id=tok.eos_token_id,
                             pad_token_id=tok.pad_token_id or tok.eos_token_id)
    return tok.decode(out[0][inp['input_ids'].shape[1]:], skip_special_tokens=True).strip()

for frase in ['Hay tocino en este sandwich.',
              'El robot se mueve hacia adelante.',
              'La fuerza es fuerte en este.']:
    print(f'{frase}\n  -> {generar(frase)}\n')

## 4. Comparar con el modelo base

Sin el adaptador, para ver qué cambió realmente el fine-tuning.

In [ ]:
with model.disable_adapter():
    for frase in ['Hay tocino en este sandwich.', 'El robot se mueve hacia adelante.']:
        print(f'{frase}\n  BASE -> {generar(frase)}\n')